# KC-UNIT Training for Massachusetts Buildings Segmentation

Semantic segmentation on the Massachusetts Buildings dataset with paired images and labels.

https://github.com/cychoi97/KC-UNIT

In [1]:
import os
import glob
import time
import csv
import random
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torch.nn.functional as F
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
PROJECT_ROOT = os.path.abspath("..")
DATASET_ROOT = os.path.join(PROJECT_ROOT, "dataset", "archive")
DATASET_VARIANT = "tiff"  # "tiff" or "png"
MODEL_SAVE_DIR = os.path.join(PROJECT_ROOT, "KC-UNIT", "models")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

IMG_SIZE = 256
NUM_CLASSES = 2
C_DIM = 1
CLASS_NAMES = ["background", "building"]
IGNORE_INDEX = None  # set to 0 to ignore background in metrics

BATCH_SIZE = 8
LEARNING_RATE = 1e-4
EPOCHS = 50

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "train")
TRAIN_MASK_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "train_labels")
VAL_IMG_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "val")
VAL_MASK_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "val_labels")
TEST_IMG_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "test")
TEST_MASK_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "test_labels")

def _count_images(path, exts):
    count = 0
    for ext in exts:
        count += len(glob.glob(os.path.join(path, f"*.{ext}")))
    return count

exts = ["tif", "tiff"] if DATASET_VARIANT == "tiff" else ["png", "jpg", "jpeg"]

for p in [TRAIN_IMG_DIR, TRAIN_MASK_DIR, VAL_IMG_DIR, VAL_MASK_DIR, TEST_IMG_DIR, TEST_MASK_DIR]:
    if not os.path.isdir(p):
        raise FileNotFoundError(f"Missing directory: {p}")

print(f"Device: {DEVICE}")
print("Dataset variant:", DATASET_VARIANT)
print("Train images:", _count_images(TRAIN_IMG_DIR, exts), "Train labels:", _count_images(TRAIN_MASK_DIR, exts))
print("Val images:", _count_images(VAL_IMG_DIR, exts), "Val labels:", _count_images(VAL_MASK_DIR, exts))
print("Test images:", _count_images(TEST_IMG_DIR, exts), "Test labels:", _count_images(TEST_MASK_DIR, exts))

Device: cuda
Dataset variant: tiff
Train images: 137 Train labels: 137
Val images: 4 Val labels: 4
Test images: 10 Test labels: 10


## Dataset Loader

In [3]:
class MassachusettsBuildingsDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size=256, variant="tiff"):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.img_size = img_size
        self.exts = ["tif", "tiff"] if variant == "tiff" else ["png", "jpg", "jpeg"]

        self.pairs = self._collect_pairs()
        if len(self.pairs) == 0:
            raise RuntimeError(f"No image/mask pairs found in {image_dir} and {mask_dir}")

        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])
        self.mask_resize = transforms.Resize((img_size, img_size), interpolation=Image.NEAREST)

    def _collect_pairs(self):
        img_paths = []
        for ext in self.exts:
            img_paths.extend(sorted(self.image_dir.glob(f"*.{ext}")))
        mask_map = {}
        for ext in self.exts:
            for p in self.mask_dir.glob(f"*.{ext}"):
                mask_map[p.stem] = p
        pairs = []
        for img_path in img_paths:
            mask_path = mask_map.get(img_path.stem)
            if mask_path is not None:
                pairs.append((img_path, mask_path))
        return pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        image = Image.open(img_path).convert("L")
        mask = Image.open(mask_path)

        image = self.img_transform(image)
        mask = self.mask_resize(mask)
        mask = np.array(mask)
        if mask.ndim == 3:
            mask = mask[..., 0]
        mask = (mask > 127).astype(np.int64)
        mask = torch.from_numpy(mask)
        return image, mask

## KC-UNIT Model (from cychoi97/KC-UNIT)

In [4]:
# Adapted from https://github.com/cychoi97/KC-UNIT (model.py)
import numpy as np
import torch.nn.utils.spectral_norm as spectral_norm


class ResidualBlock(nn.Module):
    """Residual Block with instance normalization."""
    def __init__(self, dim_in, dim_out):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(dim_in, dim_out, kernel_size=3, stride=1, padding=1, bias=False),
            nn.InstanceNorm2d(dim_out, affine=True, track_running_stats=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim_out, dim_out, kernel_size=3, stride=1, padding=1, bias=False),
            nn.InstanceNorm2d(dim_out, affine=True, track_running_stats=True),
        )

    def forward(self, x):
        return x + self.main(x)


class Generator(nn.Module):
    """Generator network."""
    def __init__(self, conv_dim=64, c_dim=8, repeat_num=6):
        super().__init__()
        layers = []
        layers.append(
            nn.Conv2d(1 + c_dim, conv_dim, kernel_size=7, stride=1, padding=3, bias=False)
        )
        layers.append(nn.InstanceNorm2d(conv_dim, affine=True, track_running_stats=True))
        layers.append(nn.ReLU(inplace=True))

        curr_dim = conv_dim
        for _ in range(2):
            layers.append(
                nn.Conv2d(curr_dim, curr_dim * 2, kernel_size=4, stride=2, padding=1, bias=False)
            )
            layers.append(
                nn.InstanceNorm2d(curr_dim * 2, affine=True, track_running_stats=True)
            )
            layers.append(nn.ReLU(inplace=True))
            curr_dim = curr_dim * 2

        for _ in range(repeat_num):
            layers.append(ResidualBlock(dim_in=curr_dim, dim_out=curr_dim))

        for _ in range(2):
            layers.append(
                nn.ConvTranspose2d(curr_dim, curr_dim // 2, kernel_size=4, stride=2, padding=1, bias=False)
            )
            layers.append(
                nn.InstanceNorm2d(curr_dim // 2, affine=True, track_running_stats=True)
            )
            layers.append(nn.ReLU(inplace=True))
            curr_dim = curr_dim // 2

        layers.append(nn.Conv2d(curr_dim, 1, kernel_size=7, stride=1, padding=3, bias=False))
        layers.append(nn.Tanh())
        self.main = nn.Sequential(*layers)

    def forward(self, x, c):
        c = c.view(c.size(0), c.size(1), 1, 1)
        c = c.repeat(1, 1, x.size(2), x.size(3))
        x = torch.cat([x, c], dim=1)
        return self.main(x)


class Discriminator(nn.Module):
    """Discriminator network with PatchGAN."""
    def __init__(self, image_size=512, conv_dim=32, c_dim=6, repeat_num=7):
        super().__init__()
        layers = []
        layers.append(nn.Conv2d(1, conv_dim, kernel_size=4, stride=2, padding=1))
        layers.append(nn.LeakyReLU(0.01))

        curr_dim = conv_dim
        for _ in range(1, repeat_num):
            layers.append(
                nn.Conv2d(curr_dim, curr_dim * 2, kernel_size=4, stride=2, padding=1)
            )
            layers.append(nn.LeakyReLU(0.01))
            curr_dim = curr_dim * 2

        kernel_size = int(image_size / np.power(2, repeat_num))
        self.main = nn.Sequential(*layers)
        self.conv1 = nn.Conv2d(curr_dim, 1, kernel_size=3, stride=1, padding=1, bias=False)
        self.conv2 = nn.Conv2d(curr_dim, c_dim, kernel_size=kernel_size, bias=False)

    def forward(self, x):
        h = self.main(x)
        out_src = self.conv1(h)
        out_cls = self.conv2(h)
        return out_src, out_cls.view(out_cls.size(0), out_cls.size(1))


class Generator_PG(nn.Module):
    """Generator network."""
    def __init__(self, conv_dim=64, c_dim=8, repeat_num=6):
        super().__init__()
        self.init = nn.Sequential(
            nn.Conv2d(1 + c_dim, conv_dim, kernel_size=7, stride=1, padding=3, bias=False),
            nn.InstanceNorm2d(conv_dim, affine=True, track_running_stats=True),
            nn.ReLU(inplace=True),
        )

        self.down1 = self.down_layer(conv_dim, kernel_size=4, stride=2, padding=1, bias=False)
        self.down2 = self.down_layer(conv_dim * 2, kernel_size=4, stride=2, padding=1, bias=False)

        residual_layers = []
        for _ in range(repeat_num):
            residual_layers.append(ResidualBlock(dim_in=conv_dim * 4, dim_out=conv_dim * 4))
        self.residual = nn.Sequential(*residual_layers)

        self.up1 = self.up_layer(conv_dim * 4, kernel_size=3, stride=1, padding=1, bias=False)
        self.up2 = self.up_layer(conv_dim * 2, kernel_size=3, stride=1, padding=1, bias=False)

        self.out = nn.Sequential(
            nn.Conv2d(conv_dim, 1, kernel_size=7, stride=1, padding=3, bias=False),
            nn.Tanh(),
        )

    def down_layer(self, conv_dim, kernel_size=4, stride=2, padding=1, bias=False):
        layer = nn.Sequential(
            nn.Conv2d(conv_dim, conv_dim * 2, kernel_size=kernel_size, stride=stride, padding=padding, bias=bias),
            nn.InstanceNorm2d(conv_dim * 2, affine=True, track_running_stats=True),
            nn.ReLU(inplace=True),
        )
        return layer

    def up_layer(self, conv_dim, kernel_size=3, stride=1, padding=1, bias=False):
        layer = nn.Sequential(
            nn.Conv2d(conv_dim, conv_dim // 2 * 4, kernel_size=kernel_size, stride=stride, padding=padding, bias=bias),
            nn.PixelShuffle(2),
            nn.InstanceNorm2d(conv_dim // 2, affine=True, track_running_stats=True),
            nn.ReLU(inplace=True),
        )
        return layer

    def forward(self, x, c):
        c = c.view(c.size(0), c.size(1), 1, 1)
        c = c.repeat(1, 1, x.size(2), x.size(3))
        x = torch.cat([x, c], dim=1)

        init = self.init(x)
        down1 = self.down1(init)
        down2 = self.down2(down1)
        residual = self.residual(down2)
        up1 = self.up1(residual)
        up2 = self.up2(up1)
        out = self.out(up2)
        dec_feature = up1
        return out, dec_feature


class Discriminator_PG(nn.Module):
    """Discriminator network with U-Net for GGDR."""
    def __init__(self, image_size=512, conv_dim=32, c_dim=6, repeat_num=7):
        super().__init__()
        self.init_down = nn.Sequential(
            spectral_norm(nn.Conv2d(1, conv_dim, kernel_size=4, stride=2, padding=1)),
            nn.LeakyReLU(0.01),
        )
        self.down1 = self.down_layer(conv_dim, kernel_size=4, stride=2, padding=1)
        self.down2 = self.down_layer(conv_dim * 2, kernel_size=4, stride=2, padding=1)
        self.down3 = self.down_layer(conv_dim * 4, kernel_size=4, stride=2, padding=1)
        self.down4 = self.down_layer(conv_dim * 8, kernel_size=4, stride=2, padding=1)
        self.down5 = self.down_layer(conv_dim * 16, kernel_size=4, stride=2, padding=1)
        self.down6 = self.down_layer(conv_dim * 32, kernel_size=4, stride=2, padding=1)

        self.init_up = nn.Sequential(
            spectral_norm(nn.Conv2d(conv_dim * 64, conv_dim * 128, kernel_size=1, stride=1, padding=0, bias=False)),
            nn.PixelShuffle(2),
            nn.LeakyReLU(0.01),
        )
        self.up1 = self.up_layer(conv_dim * 64, kernel_size=1, stride=1, padding=0, bias=False)
        self.up2 = self.up_layer(conv_dim * 32, kernel_size=1, stride=1, padding=0, bias=False)
        self.up3 = self.up_layer(conv_dim * 16, kernel_size=1, stride=1, padding=0, bias=False)
        self.up4 = self.up_layer(conv_dim * 8, kernel_size=1, stride=1, padding=0, bias=False)
        self.up5 = nn.Sequential(
            spectral_norm(nn.Conv2d(conv_dim * 4, conv_dim * 16, kernel_size=1, stride=1, padding=0, bias=False)),
            nn.PixelShuffle(2),
            nn.LeakyReLU(0.01),
        )

        kernel_size = int(image_size / np.power(2, repeat_num))
        self.conv1 = nn.Conv2d(conv_dim * 64, 1, kernel_size=3, stride=1, padding=1, bias=False)
        self.conv2 = nn.Conv2d(conv_dim * 64, c_dim, kernel_size=kernel_size, bias=False)

    def down_layer(self, conv_dim, kernel_size=4, stride=2, padding=1):
        layer = nn.Sequential(
            spectral_norm(nn.Conv2d(conv_dim, conv_dim * 2, kernel_size=kernel_size, stride=stride, padding=padding)),
            nn.LeakyReLU(0.01),
        )
        return layer

    def up_layer(self, conv_dim, kernel_size=1, stride=1, padding=0, bias=False):
        layer = nn.Sequential(
            spectral_norm(nn.Conv2d(conv_dim, conv_dim, kernel_size=kernel_size, stride=stride, padding=padding, bias=bias)),
            nn.PixelShuffle(2),
            nn.LeakyReLU(0.01),
        )
        return layer

    def forward(self, x):
        init_down = self.init_down(x)
        down1 = self.down1(init_down)
        down2 = self.down2(down1)
        down3 = self.down3(down2)
        down4 = self.down4(down3)
        down5 = self.down5(down4)
        down6 = self.down6(down5)

        init_up = self.init_up(down6)
        up1 = torch.cat([init_up, down5], dim=1)
        up1 = self.up1(up1)
        up2 = torch.cat([up1, down4], dim=1)
        up2 = self.up2(up2)
        up3 = torch.cat([up2, down3], dim=1)
        up3 = self.up3(up3)
        up4 = torch.cat([up3, down2], dim=1)
        up4 = self.up4(up4)
        up5 = torch.cat([up4, down1], dim=1)
        up5 = self.up5(up5)

        out_src = self.conv1(down6)
        out_cls = self.conv2(down6)
        out_feature = up5
        return out_src, out_cls.view(out_cls.size(0), out_cls.size(1)), out_feature

## Losses, Optimizer, and Metrics

In [5]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, probs, targets):
        if probs.dim() == 4:
            probs = probs.squeeze(1)
        if targets.dim() == 4:
            targets = targets.squeeze(1)
        probs = probs.float()
        targets = targets.float()
        probs = probs.view(probs.size(0), -1)
        targets = targets.view(targets.size(0), -1)
        intersection = (probs * targets).sum(dim=1)
        union = probs.sum(dim=1) + targets.sum(dim=1)
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()


class SegmentationMetrics:
    def __init__(self, eps=1e-7):
        self.eps = eps
        self.reset()

    def reset(self):
        self.tp = 0.0
        self.fp = 0.0
        self.fn = 0.0
        self.correct = 0.0
        self.total = 0.0

    def update(self, probs, targets):
        if probs.dim() == 4:
            probs = probs.squeeze(1)
        if targets.dim() == 4:
            targets = targets.squeeze(1)
        preds = (probs >= 0.5).long()
        targets = targets.long()
        self.correct += (preds == targets).sum().item()
        self.total += targets.numel()
        self.tp += ((preds == 1) & (targets == 1)).sum().item()
        self.fp += ((preds == 1) & (targets == 0)).sum().item()
        self.fn += ((preds == 0) & (targets == 1)).sum().item()

    def compute(self):
        precision = self.tp / (self.tp + self.fp + self.eps)
        recall = self.tp / (self.tp + self.fn + self.eps)
        f1 = 2 * precision * recall / (precision + recall + self.eps)
        iou = self.tp / (self.tp + self.fp + self.fn + self.eps)
        dice = 2 * self.tp / (2 * self.tp + self.fp + self.fn + self.eps)
        accuracy = self.correct / max(self.total, 1)
        return {
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "iou": float(iou),
            "dice": float(dice),
            "dice_loss": float(1.0 - dice),
        }


model = Generator(conv_dim=64, c_dim=C_DIM, repeat_num=6).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, factor=0.5)

criterion_bce = nn.BCELoss()
criterion_dice = DiceLoss()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 8,405,440


## DataLoaders

In [6]:
num_workers = min(4, os.cpu_count() or 1)

train_dataset = MassachusettsBuildingsDataset(
    TRAIN_IMG_DIR,
    TRAIN_MASK_DIR,
    img_size=IMG_SIZE,
    variant=DATASET_VARIANT,
)
val_dataset = MassachusettsBuildingsDataset(
    VAL_IMG_DIR,
    VAL_MASK_DIR,
    img_size=IMG_SIZE,
    variant=DATASET_VARIANT,
)
test_dataset = MassachusettsBuildingsDataset(
    TEST_IMG_DIR,
    TEST_MASK_DIR,
    img_size=IMG_SIZE,
    variant=DATASET_VARIANT,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Train samples: 137
Val samples: 4
Test samples: 10


In [7]:
def format_metrics(prefix, metrics):
    return (
        f"{prefix} acc={metrics['accuracy']:.4f} "
        f"prec={metrics['precision']:.4f} recall={metrics['recall']:.4f} "
        f"f1={metrics['f1']:.4f} dice={metrics['dice']:.4f} "
        f"iou={metrics['iou']:.4f} dice_loss={metrics['dice_loss']:.4f}"
    )


def run_epoch(loader, model, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    metrics = SegmentationMetrics()
    total_loss = 0.0
    with torch.set_grad_enabled(is_train):
        for images, masks in loader:
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True).float()
            if masks.dim() == 3:
                masks = masks.unsqueeze(1)
            if is_train:
                optimizer.zero_grad()
            c = torch.zeros(images.size(0), C_DIM, device=DEVICE)
            outputs = model(images, c)
            probs = (outputs + 1.0) / 2.0
            probs = probs.clamp(0.0, 1.0)
            loss_bce = criterion_bce(probs, masks)
            loss_dice = criterion_dice(probs, masks)
            loss = loss_bce + loss_dice
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            metrics.update(probs, masks)
    avg_loss = total_loss / max(len(loader.dataset), 1)
    return avg_loss, metrics.compute()

## Training Loop

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_metrics": [], "val_metrics": []}
best_val_loss = float("inf")
best_model_path = os.path.join(MODEL_SAVE_DIR, "kc-unit-best.pth")
train_metrics_path = os.path.join(MODEL_SAVE_DIR, "metrics_train_val.csv")
best_train_loss = float("inf")
best_train_epoch = 0

train_start = time.time()
print("Training start:", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(train_start)))

with open(train_metrics_path, "w", newline="") as metrics_file:
    writer = csv.writer(metrics_file)
    writer.writerow(["epoch", "split", "loss", "accuracy", "precision", "recall", "f1", "dice", "iou", "dice_loss"])

    for epoch in range(EPOCHS):
        epoch_start = time.time()
        train_loss, train_metrics = run_epoch(train_loader, model, optimizer=optimizer)
        val_loss, val_metrics = run_epoch(val_loader, model, optimizer=None)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_metrics"].append(train_metrics)
        history["val_metrics"].append(val_metrics)

        if train_loss < best_train_loss:
            best_train_loss = train_loss
            best_train_epoch = epoch + 1

        writer.writerow([
            epoch + 1,
            "train",
            train_loss,
            train_metrics["accuracy"],
            train_metrics["precision"],
            train_metrics["recall"],
            train_metrics["f1"],
            train_metrics["dice"],
            train_metrics["iou"],
            train_metrics["dice_loss"],
        ])
        writer.writerow([
            epoch + 1,
            "val",
            val_loss,
            val_metrics["accuracy"],
            val_metrics["precision"],
            val_metrics["recall"],
            val_metrics["f1"],
            val_metrics["dice"],
            val_metrics["iou"],
            val_metrics["dice_loss"],
        ])
        metrics_file.flush()

        print(f"Epoch {epoch + 1}/{EPOCHS} | train_loss={train_loss:.4f} val_loss={val_loss:.4f}")
        print(format_metrics("Train", train_metrics))
        print(format_metrics("Val", val_metrics))
        print(f"Epoch time: {time.time() - epoch_start:.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": best_val_loss,
                },
                best_model_path,
            )
            print(f"Saved best model: {best_model_path}")

train_end = time.time()
print("Training end:", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(train_end)))
print(f"Total training time: {train_end - train_start:.1f}s")
print(f"Best train epoch: {best_train_epoch} (train_loss={best_train_loss:.4f})")
print(f"Train/val metrics saved to {train_metrics_path}")

Training start: 2026-05-29 11:58:04
Epoch 1/50 | train_loss=1.2508 val_loss=1.2410
Train acc=0.8031 prec=0.1794 recall=0.1376 f1=0.1558 dice=0.1558 iou=0.0845 dice_loss=0.8442
Val acc=0.8913 prec=0.0000 recall=0.0000 f1=0.0000 dice=0.0000 iou=0.0000 dice_loss=1.0000
Epoch time: 4.5s
Saved best model: /media/aejaz/New_Volume/Projects/Massachusetts/KC-UNIT/models/ukan_best.pth
Epoch 2/50 | train_loss=1.1403 val_loss=1.5992
Train acc=0.8532 prec=0.3717 recall=0.1623 f1=0.2259 dice=0.2259 iou=0.1273 dice_loss=0.7741
Val acc=0.8913 prec=0.0000 recall=0.0000 f1=0.0000 dice=0.0000 iou=0.0000 dice_loss=1.0000
Epoch time: 4.3s
Epoch 3/50 | train_loss=1.0974 val_loss=1.1844
Train acc=0.8521 prec=0.4000 recall=0.2407 f1=0.3006 dice=0.3006 iou=0.1769 dice_loss=0.6994
Val acc=0.8923 prec=0.9175 recall=0.0101 f1=0.0201 dice=0.0201 iou=0.0101 dice_loss=0.9799
Epoch time: 4.3s
Saved best model: /media/aejaz/New_Volume/Projects/Massachusetts/KC-UNIT/models/ukan_best.pth
Epoch 4/50 | train_loss=1.0579 v

## Test Evaluation

In [9]:
if os.path.isfile(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best model from {best_model_path}")

test_loss, test_metrics = run_epoch(test_loader, model, optimizer=None)
print(f"Test loss: {test_loss:.4f}")
print(format_metrics("Test", test_metrics))

test_metrics_path = os.path.join(MODEL_SAVE_DIR, "metrics_test.csv")
with open(test_metrics_path, "w", newline="") as metrics_file:
    writer = csv.writer(metrics_file)
    writer.writerow(["split", "loss", "accuracy", "precision", "recall", "f1", "dice", "iou", "dice_loss"])
    writer.writerow([
        "test",
        test_loss,
        test_metrics["accuracy"],
        test_metrics["precision"],
        test_metrics["recall"],
        test_metrics["f1"],
        test_metrics["dice"],
        test_metrics["iou"],
        test_metrics["dice_loss"],
    ])
print(f"Test metrics saved to {test_metrics_path}")

Loaded best model from /media/aejaz/New_Volume/Projects/Massachusetts/KC-UNIT/models/ukan_best.pth


/tmp/ipykernel_154115/1889342327.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(best_model_path, map_location=DEVICE)


Test loss: 1.0796
Test acc=0.8290 prec=0.5796 recall=0.2990 f1=0.3945 dice=0.3945 iou=0.2457 dice_loss=0.6055
Test metrics saved to /media/aejaz/New_Volume/Projects/Massachusetts/KC-UNIT/models/metrics_test.csv


## Save Final Model

In [ ]:
final_model_path = os.path.join(MODEL_SAVE_DIR, "kc-unit-final.pth")
torch.save(model.state_dict(), final_model_path)
print(f"Final model saved to {final_model_path}")

Final model saved to /media/aejaz/New_Volume/Projects/Massachusetts/KC-UNIT/models/ukan_final.pth


In [9]:
# Run this as a cell to inspect the checkpoint
import torch
ckpt = torch.load("/media/aejaz/New Volume/Projects/Massachusetts/KC-UNIT/models/kc-unit-best.pth", map_location="cpu")
print(type(ckpt))
if isinstance(ckpt, dict):
    print("Keys:", list(ckpt.keys()))
    if "model_state_dict" in ckpt:
        keys = list(ckpt["model_state_dict"].keys())
    else:
        keys = list(ckpt.keys())
    print("First 10 weight keys:", keys[:10])

<class 'dict'>
Keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_loss']
First 10 weight keys: ['main.0.weight', 'main.1.weight', 'main.1.bias', 'main.1.running_mean', 'main.1.running_var', 'main.1.num_batches_tracked', 'main.3.weight', 'main.4.weight', 'main.4.bias', 'main.4.running_mean']


In [11]:
import torch
ckpt = torch.load("/media/aejaz/New Volume/Projects/Massachusetts/KC-UNIT/models/kc-unit-best.pth", map_location="cpu")
sd = ckpt["model_state_dict"]
for k, v in sd.items():
    print(f"{k:60s} {tuple(v.shape)}")

main.0.weight                                                (64, 2, 7, 7)
main.1.weight                                                (64,)
main.1.bias                                                  (64,)
main.1.running_mean                                          (64,)
main.1.running_var                                           (64,)
main.1.num_batches_tracked                                   ()
main.3.weight                                                (128, 64, 4, 4)
main.4.weight                                                (128,)
main.4.bias                                                  (128,)
main.4.running_mean                                          (128,)
main.4.running_var                                           (128,)
main.4.num_batches_tracked                                   ()
main.6.weight                                                (256, 128, 4, 4)
main.7.weight                                                (256,)
main.7.bias                       

In [14]:
# ============================================================
# KC-UNIT (GAN Generator) – Export 600 DPI comparison images
# ============================================================
# Run this as a NEW CELL at the bottom of train_kan-unet.ipynb
# after running all cells EXCEPT Training Loop & Test Evaluation.
#
# Cells that must already be in scope:
#   test_dataset, DEVICE
# ============================================================

from pathlib import Path
from PIL import Image, ImageDraw
import numpy as np
import torch
import torch.nn as nn

EXPORT_DPI  = 600
EXPORT_SIZE = 2048
best_model_path = "/media/aejaz/New Volume/Projects/Massachusetts/KC-UNIT/models/kc-unit-best.pth"
OUTPUT_DIR  = Path("/media/aejaz/New Volume/Projects/Massachusetts/KC-UNIT/comparison_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================
# Generator — reconstructed from checkpoint keys:
#   BatchNorm2d  (has running_mean/var/num_batches_tracked)
#   final Conv2d has bias=False  (no .bias key in checkpoint)
# =============================================================

class _ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch),
        )
    def forward(self, x):
        return x + self.main(x)


class _KCUnitGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(2, 64, 7, padding=3, bias=False),             # 0
            nn.BatchNorm2d(64),                                      # 1
            nn.ReLU(inplace=True),                                   # 2
            nn.Conv2d(64, 128, 4, stride=2, padding=1, bias=False),  # 3
            nn.BatchNorm2d(128),                                     # 4
            nn.ReLU(inplace=True),                                   # 5
            nn.Conv2d(128, 256, 4, stride=2, padding=1, bias=False), # 6
            nn.BatchNorm2d(256),                                     # 7
            nn.ReLU(inplace=True),                                   # 8
            _ResBlock(256),                                          # 9
            _ResBlock(256),                                          # 10
            _ResBlock(256),                                          # 11
            _ResBlock(256),                                          # 12
            _ResBlock(256),                                          # 13
            _ResBlock(256),                                          # 14
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1, bias=False),  # 15
            nn.BatchNorm2d(128),                                     # 16
            nn.ReLU(inplace=True),                                   # 17
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1, bias=False),   # 18
            nn.BatchNorm2d(64),                                      # 19
            nn.ReLU(inplace=True),                                   # 20
            nn.Conv2d(64, 1, 7, padding=3, bias=False),              # 21
        )

    def forward(self, x):
        return self.main(x)


# =============================================================
# Load model
# =============================================================
kc_model = _KCUnitGenerator().to(DEVICE)
ckpt     = torch.load(best_model_path, map_location=DEVICE, weights_only=False)
kc_model.load_state_dict(ckpt["model_state_dict"])
kc_model.eval()
print(f"Loaded KC-UNIT generator from {best_model_path}")

# =============================================================
# Inference
# Dataset loads images as GRAYSCALE (convert("L")) → ToTensor()
# So image_tensor is 1 x H x W  (not 3!)
# Generator input: [gray_channel, zeros]  → 2 x H x W
# =============================================================
image_tensor, mask_tensor = test_dataset[0]   # 1xHxW float32, HxW int64

gray  = image_tensor[:1]                                              # 1xHxW
zeros = torch.zeros_like(gray)                                        # 1xHxW
inp   = torch.cat([gray, zeros], dim=0).unsqueeze(0).to(DEVICE)      # 1x2xHxW

with torch.no_grad():
    logits = kc_model(inp)                    # 1x1xHxW raw logits

probs     = torch.sigmoid(logits)
pred_mask = (probs[0, 0] >= 0.5).long()      # HxW  0/1

# =============================================================
# Helpers
# =============================================================
def _to_rgb_image(tensor):
    """1xHxW or 3xHxW [0,1] tensor → HWC uint8 RGB."""
    t = tensor.detach().cpu().clamp(0, 1)
    if t.shape[0] == 1:           # grayscale → repeat to RGB for display
        t = t.repeat(3, 1, 1)
    return (t.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)

def _to_bw_mask(tensor):
    array = tensor.detach().cpu().squeeze().numpy()
    binary = array > 0.5 if array.max() <= 1 else array > 0
    return (binary.astype(np.uint8) * 255)

def _save_png_600dpi(array, path, resample):
    image = Image.fromarray(array)
    if image.size != (EXPORT_SIZE, EXPORT_SIZE):
        image = image.resize((EXPORT_SIZE, EXPORT_SIZE), resample=resample)
    image.save(path, dpi=(EXPORT_DPI, EXPORT_DPI))
    return image

def _with_title(image, title):
    rgb          = image.convert("RGB")
    title_height = 128
    canvas       = Image.new("RGB", (rgb.width, rgb.height + title_height), "white")
    canvas.paste(rgb, (0, title_height))
    draw = ImageDraw.Draw(canvas)
    draw.rectangle((0, 0, rgb.width, title_height), fill="black")
    draw.text((40, 42), title, fill="white")
    return canvas

# =============================================================
# Save
# =============================================================
stem = "test_sample"
if hasattr(test_dataset, "pairs") and test_dataset.pairs:
    stem = Path(test_dataset.pairs[0][0]).stem

input_image        = _save_png_600dpi(_to_rgb_image(image_tensor), OUTPUT_DIR / f"{stem}_input_600dpi.png",        Image.Resampling.BICUBIC)
ground_truth_image = _save_png_600dpi(_to_bw_mask(mask_tensor),    OUTPUT_DIR / f"{stem}_ground_truth_600dpi.png", Image.Resampling.NEAREST)
prediction_image   = _save_png_600dpi(_to_bw_mask(pred_mask),      OUTPUT_DIR / f"{stem}_prediction_600dpi.png",   Image.Resampling.NEAREST)

panels = [
    _with_title(input_image,        "Input"),
    _with_title(ground_truth_image, "Ground Truth"),
    _with_title(prediction_image,   "Prediction"),
]
comparison = Image.new("RGB", (sum(p.width for p in panels), max(p.height for p in panels)), "white")
x = 0
for panel in panels:
    comparison.paste(panel, (x, 0))
    x += panel.width

comparison_path = OUTPUT_DIR / f"{stem}_comparison_600dpi.png"
comparison.save(comparison_path, dpi=(EXPORT_DPI, EXPORT_DPI))

print(f"Saved input:        {OUTPUT_DIR / (stem + '_input_600dpi.png')}")
print(f"Saved ground truth: {OUTPUT_DIR / (stem + '_ground_truth_600dpi.png')}")
print(f"Saved prediction:   {OUTPUT_DIR / (stem + '_prediction_600dpi.png')}")
print(f"Saved comparison:   {comparison_path}")

Loaded KC-UNIT generator from /media/aejaz/New Volume/Projects/Massachusetts/KC-UNIT/models/kc-unit-best.pth
Saved input:        /media/aejaz/New Volume/Projects/Massachusetts/KC-UNIT/comparison_outputs/22828930_15_input_600dpi.png
Saved ground truth: /media/aejaz/New Volume/Projects/Massachusetts/KC-UNIT/comparison_outputs/22828930_15_ground_truth_600dpi.png
Saved prediction:   /media/aejaz/New Volume/Projects/Massachusetts/KC-UNIT/comparison_outputs/22828930_15_prediction_600dpi.png
Saved comparison:   /media/aejaz/New Volume/Projects/Massachusetts/KC-UNIT/comparison_outputs/22828930_15_comparison_600dpi.png
